# Assignment

Implement the following Neural Architecture Search methods using this framework:

* Random Search
* Random Sample
* Bayesian Optimisation (put some thoughts how to represent information for the surrogate
model and how to define an acquition function)
* Succesive Halving
* An iteration-based learning curve method (see, e.g., the Power-law version of [1]).

Compare the performance of these methods using some reasonable standards (e.g., give
them the same budget) on various pre-computed datasets.

Setting up NAS system

In [2]:
import os
import csv
import random

import numpy as np
import matplotlib.pyplot as plt

from nats_bench import create

SEARCH_SPACES = {'tss': 'NATS-Bench topology (TSS)'}
SEARCH_SPACE = 'tss'

FAST_MODE = True
BENCH_PATH = "data/NATS-tss-v1_0-3ffb9-simple"

DATASETS = ['cifar10', 'cifar100', 'ImageNet16-120']
DATASET = 'cifar10'
DATASET2 = 'cifar100'

if not os.path.exists(BENCH_PATH):
    raise FileNotFoundError('Benchmark not found at {}. See section 1.'.format(BENCH_PATH))

api = create(BENCH_PATH, SEARCH_SPACE, fast_mode=FAST_MODE, verbose=False)

print('Search space: %s' % SEARCH_SPACES[SEARCH_SPACE])
print('Number of architectures:', len(api))

Search space: NATS-Bench topology (TSS)
Number of architectures: 15625


# Algorithms

## Random Search

In [ ]:
def random_search(seed, hp_schedule, n_evals, dataset):
        records = []
        used_idx = []
        rng = np.random.default_rng(seed)
        best_val_accuracy = -float("inf")
        best_test_accuracy = 0
        best_idx = None
        cumulative_cost = 0.0
    
    
        if dataset == 'cifar10':
            name = 'cifar10-valid'
        else:
            name = dataset

        for i in range(n_evals):
            idx = rng.integers(0, len(api))

            # without replacement so keep doing it until its one that hasnt been tested before
            while idx in used_idx:
                idx = rng.integers(0, len(api))

            used_idx.append(idx) 
            info = api.get_more_info(int(idx), name, hp=hp_schedule, is_random=False)
            val_accuracy = info["valid-accuracy"]
            test_accuracy = info["test-accuracy"]
            if best_val_accuracy < val_accuracy:
                best_val_accuracy = val_accuracy
                best_test_accuracy = test_accuracy
                best_idx = idx

            cumulative_cost += info["train-all-time"]
            
            records.append({
                "idx": int(idx),
                "best_idx": best_idx,
                "number": i + 1,
                "val_accuracy": val_accuracy,
                "test_accuracy": test_accuracy,
                "best_val_accuracy": best_val_accuracy,
                "best_test_accuracy": best_test_accuracy,
                "computation_cost": cumulative_cost
            })
            
        return records


## Random Sampling

In [4]:
def random_sampling(seed, hp_schedule, n_evals, dataset):
    """
    Random sampling draws a single random sample from the architecture space and returns it.

    is almost the same as random search thoo so idc

    
    """
    
    
    
    
    return records

             
             

## Bayesian Optimisation

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel


OPERATIONS = [
    "none",
    "skip_connect",
    "nor_conv_1x1",
    "nor_conv_3x3",
    "avg_pool_3x3"
]

def encode_structure(idx):
    arch_str = api.arch(idx)
    structure = []
    for dst, group in enumerate(arch_str.split('+'), start=1):
            items = group.strip().strip('|').split('|')
            for src, item in enumerate(items):
                structure.append((item.split('~')[0]))

    return structure

encoder = OneHotEncoder(categories=[OPERATIONS] * 6, sparse_output=False) # categorie for each edge 
encoder.fit([encode_structure(0)]) # init encoder

def encode_architecture(idx):
    structure = encode_structure(idx)
    encoded = encoder.transform([structure])

    return encoded[0]

def UCB(mean, std, k):
    return mean + k * std

def bayesian_optimization(seed, hp_schedule, n_evals, dataset):
    # dataset init
    if dataset == 'cifar10':
        name = 'cifar10-valid'
    else:
        name = dataset

    records = []
    rng = np.random.default_rng(seed)

    #surgorate model init
    gp = GaussianProcessRegressor(
        kernel=RBF() + WhiteKernel(),
        normalize_y=True,
        random_state=seed
    )

    initial_indices = rng.choice(len(api), size=10, replace=False) # rn 10 but we can change this
    used_idx = set(int(idx) for idx in initial_indices)

    X = np.array([
         encode_architecture(int(idx))
         for idx in initial_indices
    ]) # features

    y = [] # result

    for indice in initial_indices:
        info = api.get_more_info(int(indice), name, hp=hp_schedule, is_random=False)
        y.append(info["valid-accuracy"])
        # save info for those initial

    for _ in range(n_evals - len(initial_indices)):
        gp.fit(X, y) # fitting on the surrogate model 

        # calc the rest of the indices
        candidate_indices = [
            idx for idx in range(len(api))
            if idx not in used_idx
        ]

        X_candidates =  np.array([
            encode_architecture(idx)
            for idx in candidate_indices
        ])

        mean, std = gp.predict(X_candidates, return_std=True)
        ucb = UCB(mean, std, k=2)

        position = np.argmax(ucb)
        candidate = candidate_indices[position]

        used_idx.add(candidate)
        X = np.vstack([X ,encode_architecture(candidate)])

        info = api.get_more_info(int(candidate), name, hp=hp_schedule, is_random=False)
        y.append(info["valid-accuracy"])
        # add necessary findings



    return records






# print(encode_structure(3))
# print(encode_architecture(3))
# print(encode_architecture(3).shape)

['avg_pool_3x3', 'skip_connect', 'none', 'none', 'none', 'skip_connect']
[0. 0. 0. 0. 1. 0. 1. 0. 0. 0. 1. 0. 0. 0. 0. 1. 0. 0. 0. 0. 1. 0. 0. 0.
 0. 0. 1. 0. 0. 0.]
(30,)


## Successive Halving